In [ ]:
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import max_error
import pandas as pd
import os
import joblib
from itertools import compress

sys.path.insert(0, str(Path.cwd().parent))
from neuro_bes.data import besInferenceDatapoints
from neuro_bes.preprocessing import profile_transform
from neuro_bes.postprocessing import evaluation

In [ ]:
batch_test_full_shot=[]
bes_data=besInferenceDatapoints(path=os.path.join("/home/molnarbalazs/data/BES_ML_modelling/W7X_experimental_data_from_flap/","Dataset_Na_0_we_20250513.032.h5"))
#ignore first 0.1s of shot
#bes_data.emissions=bes_data.emissions[10000:]
#bes_data.densities=bes_data.densities[10000:]
#bes_data.tags=bes_data.tags[10000:]
batch_test_full_shot.append(bes_data)

In [ ]:
# find where the emission profile contains nan in batch_test_full_shot[0]
#filter out the nan values from the batch_test_full_shot[0] emissions, densities and tags
#valid_indices = np.where(np.max(batch_test_full_shot[0].emissions,axis=1)>=50)[0]
#batch_test_full_shot[0].emissions = batch_test_full_shot[0].emissions[valid_indices]
#batch_test_full_shot[0].densities = batch_test_full_shot[0].densities[valid_indices]
#batch_test_full_shot[0].tags = [batch_test_full_shot[0].tags[i] for i in valid_indices]

In [ ]:
pipeline=joblib.load("preprocessing_pipeline_newcuration.joblib")
model=tf.keras.models.load_model("density_prediction_model_newcuration.keras")
pipeline_batch_test_full_shot=pipeline.transform(batch_test_full_shot)
for batch in pipeline_batch_test_full_shot:
    y_pred_scaled = model.predict(batch.emissions)
    y_pred_scaled=y_pred_scaled.reshape(y_pred_scaled.shape[0], -1)
    batch.densities=y_pred_scaled
batch_test_pred_full_shot=pipeline.inverse_transform(pipeline_batch_test_full_shot)

In [ ]:
bes_data_to_plot=batch_test_pred_full_shot[0]
time_instances=[float(tag[14:18]) for tag in bes_data_to_plot.tags]
r_coord=bes_data_to_plot.grid
light_max_location=r_coord[np.argmax(bes_data_to_plot.emissions,axis=1)]

In [ ]:
# plot the density and prediction profiles on a heatmap with same color scale
# set seismic colormap with white at 0 and red/blue for positive/negative values
downsampling=100
plt.figure(figsize=(12,6))
plt.subplot(2,1,1)
# colorbar with same scale for both plots
im = plt.pcolormesh(time_instances[::downsampling], r_coord, bes_data_to_plot.emissions[::downsampling].T, cmap='RdBu_r')
cbar = plt.colorbar(im)
cbar.ax.set_ylabel('mV',fontsize=18, labelpad=10)
cbar.ax.tick_params(labelsize=12)
vmax = np.max(bes_data_to_plot.emissions)
im.set_clim(0, vmax)
plt.title('Light Profiles',fontsize=20)
plt.xlabel('Time',fontsize=18, labelpad=10)
plt.ylabel('Major radius (m)',fontsize=18, labelpad=10)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.subplot(2,1,2)
im2 = plt.pcolormesh(time_instances[::downsampling],r_coord, bes_data_to_plot.densities[::downsampling].T*1e19, cmap='RdBu_r')
cbar = plt.colorbar(im2)
cbar.ax.set_ylabel('$m^{-3}$',fontsize=18, labelpad=10)
cbar.ax.tick_params(labelsize=16)
vmax = np.max(bes_data_to_plot.densities*1e19)
im2.set_clim(0, vmax)
plt.title('Predicted Density Profiles',fontsize=20)
plt.xlabel('Time (s)',fontsize=18, labelpad=10)
plt.ylabel('Major radius (m)',fontsize=18, labelpad=10)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.tight_layout()
#plt.savefig("time_evolution_shot_"+bes_data_to_plot.ID+".png", dpi=600, bbox_inches='tight')
plt.show()
